# Tarea 1. Desigualdad de viaje al trabajo en Santiago

**Curso:** INF-497 · Análisis de Datos Espaciales · UTFSM
**Profesora:** Daniela Opitz
**Inicio en clase:** miércoles 27 mayo 2026
**Entrega:** miércoles 24 junio 2026, 23:59
**Modalidad:** grupos de 2 (o individual)

---

## Contexto y pregunta central

En Santiago el ingreso presenta una distribución desigual, y también lo hace el tiempo que las personas dedican a desplazarse al trabajo. Esta dimensión de la desigualdad, conocida en la literatura como *transport poverty*, no suele aparecer en las estadísticas oficiales, aunque se observa con claridad al comparar el tiempo medio de viaje entre distintas zonas del área metropolitana.

En esta tarea trabajan con **zonas EOD** (~560 polígonos submunicipales del Gran Santiago) y usan **distritos censales** (~300 grupos efectivos) como nivel de agrupación intermedio. La comuna no se usa como unidad analítica: la escala submunicipal es la que permite observar la heterogeneidad real del fenómeno.

> **Pregunta central:** ¿qué predice el tiempo de viaje al trabajo de una zona: la educación, el ingreso, el modo de transporte, o la estructura espacial del territorio?

## Datos

- `datos/external/eod_stgo/origenes_viajes.gpkg`: 65 591 puntos de viaje (EOD Santiago 2012). 27 622 personas con `IngresoFinal`, `Estudios`, `Ocupacion`, `ModoPriPub`, `FactorPersona`, **`Proposito`**, **`TiempoViaje`**.
- `datos/external/eod_stgo/zonas_eod.gpkg`: 803 zonas con `ZONA777`, `COMUNA`, `MACROZONA`, **`DIST_ID`** (par único `COMUNA + COD_DISTRI` calculado por spatial join con `DISTRITO_C17`).
- `datos/external/eod_stgo/Tablas_parametros/`: codebook oficial EOD (61 archivos con códigos y descripciones).

CRS de trabajo: **UTM 19S (EPSG:32719)** para todo cálculo de vecindad y distancia.

## Calendario en clases

| Fecha | Sem | Bloque T1 |
|---|---|---|
| **27 may** | 13 | Parte A: carga, agregación, mapas, W |
| **3 jun**  | 14 | Secciones 2 + 3: Moran/LISA + Theil descomposición |
| ~~10 jun~~ | 15 | Presentación proyecto (no se trabaja T1) |
| **17 jun** | 16 | Secciones 4 + 5: regionalización + OLS/diagnóstico + SAR/SEM/SLX |
| **24 jun** | 17 | **Control individual** · **entrega del notebook 23:59** |

Tienen **3 clases efectivas** para trabajar T1 en sala (27 may, 3 jun, 17 jun). Las clases sirven como guía y para resolver dudas, pero pueden avanzar en casa todo lo que necesiten entre clases y, especialmente, entre el 17 y el 24 de junio para terminar y revisar el notebook en grupo.

## Evaluación

La nota T1 se compone de dos partes:

```
Nota T1 = 0.6 × Notebook + 0.4 × Control individual
```

- **Notebook (60 %):** trabajo grupal entregable (este archivo).
- **Control individual (40 %):** prueba escrita en clase del 24 jun, a libro abierto: pueden traer su notebook ejecutado y una hoja A4 manuscrita con apuntes propios. Evalúa comprensión individual de los métodos y de los resultados de su grupo.

## Entregable

**Notebook ejecutado** `tarea1_apellido1_apellido2.ipynb`, con todas las celdas corridas, mapas visibles, e **interpretaciones escritas en celdas markdown** (no en celdas de código). Cada sección tiene una celda "Interpretación" que se evalúa como parte de la nota. La síntesis del análisis va al final del propio notebook, en celdas markdown (Sección 6).

## Rúbrica del Notebook (70 pts)

| Sección | Pts |
|---|---|
| Parte A · Preparación y visualización | 10 |
| Sección 2 · Moran + LISA + interpretación | 10 |
| Sección 3 · Desigualdad + descomposición Theil | 10 |
| Sección 4 · Regionalización | 10 |
| **Sección 5 · Regresión espacial** | **20** |
| Síntesis final (Sección 6) · MAUP, limitaciones, política | 10 |

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from libpysal import weights
from esda.moran import Moran, Moran_Local
from inequality.gini import Gini
from inequality.theil import Theil, TheilD
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler
from splot.esda import lisa_cluster, plot_moran
from spreg import OLS, ML_Lag, ML_Error

sns.set_context("notebook")
np.random.seed(42)
RUTA_DATOS = "../datos/external/eod_stgo"


---
# Parte A — Preparación (clase 27 mayo, segundo bloque)

### 1. Cargar datos
1.1 Carguen `origenes_viajes.gpkg` como `pts` y `zonas_eod.gpkg` como `zonas`.
1.2 Verifiquen CRS UTM 19S, número de puntos, personas, viajes al trabajo, zonas, distritos.

In [2]:
# Tu código aquí
pts = gpd.read_file(".\eod_stgo\origenes_viajes.gpkg")
zonas = gpd.read_file(".\eod_stgo\zonas_eod.gpkg")

In [3]:
# Verificando CRS UTM 19S
print(pts.crs, zonas.crs)

EPSG:32719 EPSG:32719


In [4]:
# Número de puntos
print(len(pts))

65591


In [5]:
# Número de Personas
print(pts["Persona"].nunique())

27622


In [6]:
# Número de viajes "Al trabajo"
print(len(pts[pts['Proposito'] == "Al trabajo"]))

12592


In [7]:
# Número de Zonas
print(pts["ZonaOrigen"].nunique())

847


In [8]:
# Número de Distritos
print(zonas["DIST_ID"].nunique())

340


### 2. Agregar microdatos a zonas
Construyan un `GeoDataFrame` `gdf` (una fila por zona) con:

**Variable dependiente:**
- `tiempo_trabajo_medio` — media del `TiempoViaje` filtrando `Proposito='Al trabajo'` y tiempos entre 1 y 180 min.

**Predictores:**
- `ingreso_medio`, `pct_educ_sup` (≥9 = CFT+IP+Univ), `pct_modo_publico` (denominador todos los viajes), `edad_media`.

**Control:**
- `n_personas`, `n_viajes_trabajo` — filtrar zonas con `n_personas ≥ 5 and n_viajes_trabajo ≥ 5`.

Recomendaciones: deduplicar por `Persona` para atributos personales; excluir `Estudios ∈ {98, 99}`; ponderar atributos personales por `FactorPersona`.

In [33]:
pts_persona = pts.drop_duplicates(subset='Persona').copy()
filter_pts_persona = pts_persona[(pts_persona['Estudios'] != 98) & (pts_persona['Estudios'] != 99)]

In [ ]:
w  = filter_pts_persona["FactorPersona"]
ingreso_con_factor = (filter_pts_persona["IngresoFinal"] * w).sum() / w.sum()

np.float64(271656.80304598063)

In [62]:
# 1. Filtrar el DataFrame con tus condiciones
filtro = (pts['Proposito'] == 'Al trabajo') & (pts['TiempoViaje'] < 180) & (pts['TiempoViaje'] > 1)
pts_filtrado = pts[filtro]

# 2. Agrupar por 'Zona', seleccionar 'TiempoViaje' y calcular la media
tiempo_trabajo_medio = pts_filtrado.groupby('Zona')['TiempoViaje'].mean()

# Mostrar el resultado
print(tiempo_trabajo_medio)

Zona
1      32.291667
2      48.571429
3      39.833333
4      35.571429
6      43.000000
         ...    
857    64.218750
858    56.850000
859    61.000000
860    64.180328
861    70.000000
Name: TiempoViaje, Length: 736, dtype: float64


In [77]:
filter_pts_persona['educ_sup'] = (filter_pts_persona['Estudios'] >= 9).astype(int)

In [ ]:
#- `ingreso_medio`, `pct_educ_sup` (≥9 = CFT+IP+Univ), `pct_modo_publico` (denominador todos los viajes), `edad_media`.
#Estudios
pct_educ_sup = filter_pts_persona.groupby['Zona'].apply(lambda x: (x['educ_sup'] * x['FactorPersona']).sum() / x['FactorPersona'].sum()) * 100
display(filter_pts_persona['educ_sup'])

TypeError: 'method' object is not subscriptable

In [13]:
gdf = pd.merge(pts_persona, zonas, left_on='Zona', right_on='ZONA777', how='inner')

In [14]:
display(gdf)

,Hogar,Persona,Viaje,Etapas,ComunaOrigen,ComunaDestino,SectorOrigen,SectorDestino,ZonaOrigen,ZonaDestino,...,geometry_x,ZONA777,COMUNA,MACROZONA,AREA,DIST_ID,COMUNA_C17,NOM_COMUNA,COD_DISTRI,geometry_y
0,173431,17343102,1734310202,1,94.0,94.0,2.0,2.0,400,407,...,POINT (335208.719 6288387),407,P.A. CERDA,SUR,170.78125,13121_4,13121,PEDRO AGUIRRE CERDA,4,"POLYGON ((343682.805 6292305.71, 343803.098 62..."
1,173431,17343101,1734310101,2,94.0,328.0,2.0,2.0,407,126,...,POINT (338812.281 6292391),407,P.A. CERDA,SUR,170.78125,13121_4,13121,PEDRO AGUIRRE CERDA,4,"POLYGON ((343682.805 6292305.71, 343803.098 62..."
2,173441,17344101,1734410101,2,94.0,71.0,2.0,3.0,407,307,...,POINT (338536.438 6291928),407,P.A. CERDA,SUR,170.78125,13121_4,13121,PEDRO AGUIRRE CERDA,4,"POLYGON ((343682.805 6292305.71, 343803.098 62..."
3,173441,17344103,1734410301,2,94.0,91.0,2.0,3.0,407,437,...,POINT (338536.438 6291928),407,P.A. CERDA,SUR,170.78125,13121_4,13121,PEDRO AGUIRRE CERDA,4,"POLYGON ((343682.805 6292305.71, 343803.098 62..."
4,173462,17346201,1734620101,1,94.0,94.0,2.0,2.0,398,398,...,POINT (334334.094 6292455),398,EL BOSQUE,SUR,592.37500,13105_8,13105,EL BOSQUE,8,"POLYGON ((343759.442 6283853.918, 343932.589 6..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25618,329271,32927102,3292710201,1,96.0,92.0,5.0,3.0,180,281,...,POINT (344872.656 6287845),180,PROVIDENCIA,ORIENTE,88.56250,13123_3,13123,PROVIDENCIA,3,"POLYGON ((350700.9 6301032.811, 350867.346 630..."
25619,329271,32927103,3292710301,1,96.0,328.0,5.0,2.0,180,129,...,POINT (344872.656 6287845),180,PROVIDENCIA,ORIENTE,88.56250,13123_3,13123,PROVIDENCIA,3,"POLYGON ((350700.9 6301032.811, 350867.346 630..."
25620,329271,32927104,3292710401,1,96.0,96.0,5.0,5.0,180,180,...,POINT (344872.656 6287845),180,PROVIDENCIA,ORIENTE,88.56250,13123_3,13123,PROVIDENCIA,3,"POLYGON ((350700.9 6301032.811, 350867.346 630..."
25621,329271,32927105,3292710501,1,96.0,95.0,5.0,5.0,180,643,...,POINT (344872.656 6287845),180,PROVIDENCIA,ORIENTE,88.56250,13123_3,13123,PROVIDENCIA,3,"POLYGON ((350700.9 6301032.811, 350867.346 630..."


### 3. Visualización (Actividad 3)
**3.A** Coropletas de la variable dependiente y los 3 predictores principales con **Fisher-Jenks k=5**.

**3.B** Comparación lado a lado: `tiempo_trabajo_medio` a nivel **zona EOD** vs re-agregado a **distrito censal**. Este ejercicio ilustra el **MAUP (*Modifiable Areal Unit Problem*)**: el resultado de un análisis espacial puede cambiar cuando se modifica la unidad de agregación (zona vs distrito) o sus límites, aun cuando los datos subyacentes son los mismos.

In [15]:
# 3.A — coropletas


In [16]:
# 3.B — comparación zona vs distrito


**Interpretación 3.B (MAUP):** *(escriban qué se pierde al pasar a distrito)*

### 4. Matrices de pesos espaciales (Actividad 2)
Construyan `Queen` y `KNN k=8`. Reporten islas, componentes, cardinalidad. Justifiquen cuál usan.

In [17]:
# Tu código aquí


**Justificación elección de W:** *(escriban acá)*

---
# Parte B — Análisis

## Sección 2 — Moran global + LISA (clase 3 jun) · 10 pts
**2.A** Moran I del `tiempo_trabajo_medio` y del `ingreso_medio` (para comparar las dos dimensiones de desigualdad). Gráfico de Moran para tiempo.
**2.B** LISA del `tiempo_trabajo_medio`. Mapa de cuadrantes HH/LL/LH/HL significativos (α=0.05).
**2.C** Identifiquen al menos **3 clusters HH y 3 LL** nombrando las zonas y/o comunas en que se ubican. Comenten cómo se relaciona el patrón observado con la estructura urbana del Gran Santiago.

In [18]:
# 2.A — Moran global


In [19]:
# 2.B — LISA


**Interpretación 2.C (clusters reales):** *(listen barrios, comuniquen el patrón espacial)*

## Sección 3 — Desigualdad espacial (clase 3 jun) · 10 pts
**3.A** Gini y Theil del `tiempo_trabajo_medio` a nivel zona EOD. Comparen con el Gini del ingreso (¿cuál es más desigual?).
**3.B** Gini calculado a nivel distrito y compárenlo con el de zona EOD.
**3.C** Apliquen `TheilD` usando `DIST_ID` como variable de grupo. Reporten % entre vs intra distrito.

In [20]:
# Tu código aquí


**Interpretación 3.C:** ¿qué fracción de la desigualdad de tiempo es estructural (entre distritos) vs individual (intra distrito)? ¿Cómo se compara con el ingreso?

## Sección 4 — Regionalización (clase 17 jun) · 10 pts
**4.A** Estandaricen 4 variables (`tiempo_trabajo_medio`, `ingreso_medio`, `pct_educ_sup`, `pct_modo_publico`). K-Means k=6 (`random_state=42`) y AHC k=6 con restricción Queen.
**4.B** Mapeen ambos resultados lado a lado.
**4.C** Caractericen las 6 regiones AHC con la media de cada variable. Nombren cada región.

In [21]:
# 4.A — clustering


In [22]:
# 4.B — mapas


In [23]:
# 4.C — caracterización


**Interpretación 4.C:** *(nombren cada región con un perfil sustantivo)*

## Sección 5 — Regresión espacial (clase 17 jun) · 20 pts

Modelo: `tiempo_trabajo_medio ~ ingreso_medio + pct_educ_sup + pct_modo_publico + edad_media`.

**5.A** OLS con `spreg.OLS(..., spat_diag=True, moran=True)`. Reporten R², AIC, Moran residuos, LM-lag, LM-error y robustos. Reporten también la tabla de coeficientes con sus p-values.
**5.B** ¿Corresponde SAR o SEM? Justifiquen.
**5.C** Ajusten SAR (`ML_Lag`) y SEM (`ML_Error`). Tabla comparativa OLS / SAR / SEM en AIC, pseudo R², rho/lambda.
**5.D** Ajusten SLX con rezagos espaciales de los predictores. Interpreten al menos un coeficiente spillover sustantivamente.

> Lo que no alcancen a cerrar en clase queda como trabajo entre el 17 y el 24 de junio.

In [24]:
# 5.A — OLS + diagnóstico


**Interpretación 5.B:** *(qué modelo corresponde y por qué)*

**Interpretación adicional:** ¿el `ingreso_medio` es significativo en el OLS una vez controlados educación y modo? ¿Qué les dice esto sobre el constructo 'ingreso'?

In [25]:
# 5.C — SAR y SEM


In [26]:
# 5.D — SLX


**Interpretación 5.D (spillover):** *(escriban la lectura sustantiva de un coeficiente W_*)*

---
## 6. Síntesis final · 10 pts

En esta sección sintetizan los hallazgos del análisis. Respondan **cada pregunta en una celda markdown separada**, en 3-6 frases, citando cifras concretas de su propio análisis.

### 6.1 ¿Qué fracción del tiempo de viaje se explica?
Comparen el R² del OLS con el `rho` del SAR. ¿Qué les dice esa comparación sobre la naturaleza del fenómeno?

### 6.2 Patrón espacial de los tiempos altos
Caractericen los clusters LISA HH que encontraron: ubicación, perfil socioeconómico promedio, modo de transporte predominante. Citen al menos 2 zonas específicas con su ZONA777.

### 6.3 MAUP — Modifiable Areal Unit Problem
El **MAUP** es el problema clásico del análisis espacial según el cual los resultados estadísticos pueden cambiar cuando se modifica la **unidad de agregación** (por ejemplo, zona EOD vs distrito censal) o sus **límites**, aun cuando los datos subyacentes son los mismos.

Comparen un resultado (LISA o Theil) a las dos escalas submunicipales (**zona EOD vs distrito**). ¿Cambia la lectura del fenómeno al cambiar la escala de agregación? ¿Qué se pierde y qué se gana?

### 6.4 Política pública
Si fueran asesores del MTT, ¿qué intervención priorizarían a partir de sus resultados? Justifiquen con **una cifra** de su análisis y reconozcan **una limitación** de los datos (por ejemplo, que la EOD es de 2012 y es anterior al Metro L6 y a la pandemia).

**6.1** *(escriban acá)*

**6.2** *(escriban acá)*

**6.3** *(escriban acá)*

**6.4** *(escriban acá)*